# Day 15: Pandas 核心 —— groupby、agg、merge、concat 习题

> **范围**: 分组聚合、多聚合、transform、merge、concat、filter
> **数据**: `../data/sales.csv` + `../data/customers.csv`
> **建议用时**: 60-90 分钟
> **提示**: 注意 #48 —— Pandas 方法返回新对象，要原地修改必须显式赋值

## Easy

**1. groupby 基础 —— 按国家分组统计**

基于 `df = pd.read_csv("../data/sales.csv")`：
- 按 `country` 分组，计算每个国家的 `total` 之和
- 按 `country` 分组，计算每个国家的 `quantity` 平均值
- 按 `category` 分组，计算每个品类的订单数量（提示：`.size()` 或 `.count()`）
- 按 `country` 分组，计算每个国家的最大订单额（`total.max()`）

In [62]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/sales.csv")

print(df.groupby('country')['total'].sum())

print(df.groupby('country')['quantity'].mean())

print(df.groupby('category').size())

print(df.groupby('country')['total'].max())

country
China       80101
France     208165
Germany    144020
UK         534817
US         300613
Name: total, dtype: int64
country
China      3.093750
France     2.937500
Germany    2.727273
UK         2.959391
US         3.096000
Name: quantity, dtype: float64
category
Accessory    180
Audio         67
Computer     160
Mobile        93
dtype: int64
country
China      6495
France     9995
Germany    9995
UK         9995
US         9995
Name: total, dtype: int64


**2. agg 多聚合 —— 同时求多个统计量**

基于 `df`：
- 按 `country` 分组，对 `total` 同时求 `sum`、`mean`、`count`、`std`
- 按 `category` 分组，对 `total` 求 `sum` 和 `mean`，对 `quantity` 求 `mean` 和 `max`
- 按 `country` 分组，对 `total` 用自定义聚合函数求「极差」（max - min）
  （提示：`.agg([('range', lambda x: x.max() - x.min())])`）

In [63]:
print(df.groupby('country')['total'].agg(['sum', 'mean', 'count', 'std']))

print(df.groupby('category').agg(
    {
        'total': ['sum', 'mean'],
        'quantity': ['mean', 'max']
    }
))

print(df.groupby('country')['total'].agg([('range', lambda x: x.max() - x.min())]))

            sum         mean  count          std
country                                         
China     80101  2503.156250     32  1842.153755
France   208165  2602.062500     80  2387.795129
Germany  144020  2182.121212     66  2254.833895
UK       534817  2714.807107    197  2662.263070
US       300613  2404.904000    125  2280.839611
            total               quantity    
              sum         mean      mean max
category                                    
Accessory  441266  2451.477778  2.966667   5
Audio      204686  3055.014925  3.194030   5
Computer   363933  2274.581250  2.918750   5
Mobile     257831  2772.376344  2.892473   5
         range
country       
China     6396
France    9896
Germany   9896
UK        9896
US        9896


**3. transform —— 分组变换**

基于 `df`：
- 计算每个订单的 `total` 相对于其国家均值的偏差（`total - 国家均值`）
  （提示：`df.groupby('country')['total'].transform('mean')`）
- 创建新列 `country_rank`：每个订单在其国家内的 `total` 排名（降序，第1名=最大）
  （提示：`.transform('rank', ascending=False)`）
- 验证 `country_rank` 的 dtype 和取值范围（每个国家应该有 1~N 的整数排名）
- 人为制造 5 个 `total` 缺失值，用 `country` 分组均值填充，存为新列 `total_filled_transform`

In [64]:
print(df.groupby('country')['total'].transform('mean'))

df['country_rank'] = df.groupby('country')['total'].transform('rank', ascending=False, method='first').astype(int)
print(df['country_rank'].describe())

df.loc[df.sample(5).index, "total"] = np.nan
df["total_filled_transform"] = df["total"].fillna(
    df.groupby("country")["total"].transform("mean")
)

0      2182.121212
1      2404.904000
2      2404.904000
3      2404.904000
4      2602.062500
          ...     
495    2714.807107
496    2714.807107
497    2602.062500
498    2602.062500
499    2714.807107
Name: total, Length: 500, dtype: float64
count    500.000000
mean      66.714000
std       50.740649
min        1.000000
25%       25.750000
50%       55.000000
75%       99.000000
max      197.000000
Name: country_rank, dtype: float64


## Medium

**4. merge —— 两表关联**

读取 `customers.csv`：
- 用 `pd.merge` 将 `sales` 和 `customers` 按 `customer_id` 关联，使用 `how='left'`
- 检查有多少订单的 `name` 是 NaN（未匹配到客户）
- 统计每个客户（`customer_id` + `name`）的总消费金额（`total` 之和）
  （提示：merge 后 groupby `customer_id` 和 `name`）
- 找出总消费最高的客户（客户名 + 金额）

In [65]:
customers = pd.read_csv("../data/customers.csv")

left = pd.merge(df, customers, on='customer_id', how='left')

print(left[left["name"].isnull()])

print(left.groupby(['customer_id', 'name'])['total'].sum())

print(left.groupby(['customer_id', 'name'])['total'].sum().idxmax())

Empty DataFrame
Columns: [order_id, customer_id, product, category, quantity, price, order_date, country_x, total, country_rank, total_filled_transform, name, country_y, signup_date]
Index: []
customer_id  name   
C001         Alice      180003.0
C002         Bob        131979.0
C003         Charlie    160212.0
C004         David      143096.0
C005         Eva        131234.0
C006         Frank      198806.0
C007         Grace      143917.0
C008         Henry      164484.0
Name: total, dtype: float64
('C006', 'Frank')


**5. concat —— 拼接数据**

基于 `df`：
- 把 `df` 分成前200行和后300行两部分，用 `pd.concat` 纵向拼接回500行，用 `ignore_index=True`
- 验证拼接后的行数和列数
- 从 `df` 提取 `order_id`+`total` 和 `order_id`+`country`+`category` 两个子集，用 `pd.concat(axis=1)` 横向拼接
- 观察结果，是否有重复列？

In [66]:
df1= df.iloc[:200,:]
df2= df.iloc[200:,:]
concat = pd.concat([df1, df2], axis=0, ignore_index=True)
print(concat.shape)

df_left = df[['order_id', 'total']]
df_right = df[['order_id', 'country', 'category']]
concat2 = pd.concat([df_left, df_right], axis=1)
print(concat2.head())

(500, 11)
  order_id   total order_id  country   category
0    O1000  2598.0    O1000  Germany  Accessory
1    O1001    99.0    O1001       US  Accessory
2    O1002   396.0    O1002       US   Computer
3    O1003   396.0    O1003       US      Audio
4    O1004   495.0    O1004   France     Mobile


**6. 分组筛选 —— filter**

基于 `df`：
- 用 `groupby + filter` 保留订单数量 > 50 的国家
- 用 `groupby + filter` 保留平均订单额 > 2000 的品类
- 用 `groupby + filter` 保留 `total` 最大值 >= 8000 的国家
- 对比 filter 和直接布尔筛选的区别：filter 保留的是「整个组」，布尔筛选保留的是「满足条件的行」

In [67]:
print((df.groupby('country').filter(lambda x: len(x) > 50))['country'].unique())

print(df.groupby('category').filter(lambda x: x['total'].mean() > 2000)['category'].unique())

print(df.groupby('country').filter(lambda x: x['total'].max() >= 8000)['country'].unique())

['Germany' 'US' 'France' 'UK']
['Accessory' 'Computer' 'Audio' 'Mobile']
['Germany' 'US' 'France' 'UK']


**7. 双重分组 —— country + category**

基于 `df`：
- 按 `country` 和 `category` 双重分组，求 `total` 的 `sum` 和 `mean`
- 找出「UK 的 Computer 品类」总销售额（从结果中定位）
- 用 `.reset_index()` 把双重分组结果变成普通 DataFrame
- 对上述结果按 `total` 的 `sum` 降序排列，查看前10名

In [68]:
result = df.groupby(['country', 'category'])['total'].agg(['sum', 'mean'])
print(result)

print(result.loc['UK', 'Computer'])

print(result.reset_index())

print(result.reset_index().sort_values(by='sum', ascending=False).head(10))

                        sum         mean
country category                        
China   Accessory   21271.0  2363.444444
        Audio        8191.0  4095.500000
        Computer    32159.0  2297.071429
        Mobile      18480.0  2640.000000
France  Accessory   53936.0  2451.636364
        Audio       60245.0  3543.823529
        Computer    51431.0  2236.130435
        Mobile      41554.0  2444.352941
Germany Accessory   45042.0  2047.363636
        Audio       18975.0  2108.333333
        Computer    51036.0  2218.956522
        Mobile      28472.0  2588.363636
UK      Accessory  164595.0  2318.239437
        Audio       73734.0  3511.142857
        Computer   158700.0  2333.823529
        Mobile     131293.0  3647.027778
US      Accessory  153425.0  2841.203704
        Audio       43541.0  2418.944444
        Computer    66609.0  2148.677419
        Mobile      31042.0  1552.100000
sum     158700.000000
mean      2333.823529
Name: (UK, Computer), dtype: float64
    country   cat

## Hard

**8. 综合 —— merge + groupby + agg + pivot_table**

基于 `df` 和 `customers.csv`：
1. LEFT JOIN `df` 和 `customers`（`customer_id`）
2. 按 `country` 和 `name` 分组，求每个客户的：总消费、平均订单额、订单数、最大订单额
3. 对上述结果用 `pivot_table`：行=country，值=总消费，聚合=sum
4. 找出每个国家消费最高的客户（提示：先按 country 分组，再用 `idxmax` 或 `rank`）
5. 把结果写入 `customer_summary.csv`（index=False）

In [69]:
left = pd.merge(df, customers, on='customer_id', how='left')

left_filtered = left.groupby(['country_x', 'name']).agg(
    total_sum=("total", "sum"),
    total_mean=("total", "mean"),
    order_count=("quantity", "count"),  
    quantity_max=("quantity", "max")
).reset_index()
print(left_filtered)

pivot = pd.pivot_table(left_filtered, index='country_x', values='total_sum', aggfunc='sum', fill_value=0).reset_index()
print(pivot)

top_customer = left_filtered[
    left_filtered.groupby("country_x")["total_sum"].rank(ascending=False, method="min") == 1
]
print(top_customer)

top_customer.to_csv('customer_summary.csv', index=False)


   country_x     name  total_sum   total_mean  order_count  quantity_max
0      China    Alice     9290.0  2322.500000            4             4
1      China      Bob     1295.0   647.500000            2             4
2      China  Charlie    13981.0  2330.166667            6             5
3      China    David     4690.0  1563.333333            3             5
4      China      Eva     9490.0  4745.000000            2             5
5      China    Frank    13587.0  2717.400000            5             5
6      China    Grace    14985.0  3746.250000            4             5
7      China    Henry    12783.0  2130.500000            6             5
8     France    Alice    43155.0  2877.000000           15             4
9     France      Bob    17785.0  2223.125000            8             3
10    France  Charlie    25974.0  3246.750000            8             5
11    France    David    26868.0  2442.545455           12             5
12    France      Eva    14973.0  1663.666667      

**9. 客户分析 —— RFM-like 分析**

基于 `df`（无需 merge）：
- 按 `customer_id` 分组，计算每个客户的：
  - `R`（Recency）= 最后下单日期距今天数（假设今天是 2024-06-01，提示：`(2024-06-01 - 最后日期).days`）
  - `F`（Frequency）= 订单数量
  - `M`（Monetary）= 总消费金额
- 用 `pd.qcut` 对 R、F、M 分别分3箱（低/中/高），注意 R 越小越好（所以标签要反向）
- 计算每个客户的 RFM 综合得分（简单相加：低=1，中=2，高=3）
- 找出 RFM 得分最高的前5名客户

In [70]:
df["order_date"] = pd.to_datetime(df["order_date"])  
today = pd.to_datetime("2024-06-01")

rfm_raw = df.groupby('customer_id').agg(
    last_order_date=('order_date', 'max'),
    F=('order_date', 'count'),
    M=('total', 'sum')
).reset_index()

rfm_raw['R'] = (today - rfm_raw['last_order_date']).dt.days
print(rfm_raw.head())

rfm_raw['R_score'] = pd.qcut(rfm_raw['R'], 3, labels=[3, 2, 1])
rfm_raw['F_score'] = pd.qcut(rfm_raw['F'], 3, labels=[1, 2, 3])
rfm_raw['M_score'] = pd.qcut(rfm_raw['M'], 3, labels=[1, 2, 3])

rfm_raw['RFM_Score'] = rfm_raw['R_score'].astype(str) + rfm_raw['F_score'].astype(str) + rfm_raw['M_score'].astype(str)
top5_customers = rfm_raw.sort_values(by='RFM_Score', ascending=False).head(5)
print(top5_customers)


  customer_id last_order_date   F         M    R
0        C001      2024-12-30  71  180003.0 -212
1        C002      2024-12-02  46  131979.0 -184
2        C003      2024-12-28  64  160212.0 -210
3        C004      2024-12-31  76  143096.0 -213
4        C005      2024-12-27  51  131234.0 -209
  customer_id last_order_date   F         M    R R_score F_score M_score  \
0        C001      2024-12-30  71  180003.0 -212       3       3       3   
3        C004      2024-12-31  76  143096.0 -213       3       3       1   
2        C003      2024-12-28  64  160212.0 -210       3       2       2   
6        C007      2024-12-28  58  143917.0 -210       3       1       2   
4        C005      2024-12-27  51  131234.0 -209       2       1       1   

  RFM_Score  
0       333  
3       331  
2       322  
6       312  
4       211  


参考答案

题9 · R 为负数 + 字符串拼接 —— #51/#52

In [72]:
# ❌ today 在订单日期之前，R 为负数
today = pd.to_datetime("2024-06-01")  # 订单日期是 2024-01~12
rfm_raw['R'] = (today - rfm_raw['last_order_date']).dt.days  # → -212 天

# ✅ 正确
today = pd.to_datetime("2025-01-01")  # 设在所有订单日期之后

# ❌ 字符串拼接
rfm_raw['RFM_Score'] = rfm_raw['R_score'].astype(str) + rfm_raw['F_score'].astype(str) + rfm_raw['M_score'].astype(str)  # → "333" 字符串

# ✅ 数值相加
rfm_raw['RFM_Score'] = rfm_raw['R_score'].astype(int) + rfm_raw['F_score'].astype(int) + rfm_raw['M_score'].astype(int)  # → 9

**10. 综合管道 —— 从多表到报告**

写一段完整脚本：

**阶段1 —— 读取多表**:
- 读取 `../data/sales.csv` 和 `../data/customers.csv`

**阶段2 —— 关联**:
- LEFT JOIN 两表，保留所有订单
- 检查未匹配的订单数和比例

**阶段3 —— 分组分析**:
- 按 `country` 分组：`total` 的 `sum`、`mean`、`count`
- 按 `category` 分组：`total` 的 `sum`、`mean`，`quantity` 的 `mean`
- 按 `country` + `category` 双重分组：`total` 的 `sum`

**阶段4 —— 输出**:
- 把三个分组结果分别写入三个 CSV：
  - `country_stats.csv`
  - `category_stats.csv`
  - `country_category_stats.csv`
- 把 JOIN 后的完整数据写入 `sales_with_customers.csv`

**阶段5 —— 验证**:
- 打印每个输出文件的行数
- 确认 `sales_with_customers.csv` 的列数 = 原 sales 列数 + customers 列数 - 1（去重 customer_id）

In [71]:
sales = pd.read_csv("../data/sales.csv")
customers = pd.read_csv("../data/customers.csv")

left = pd.merge(sales, customers, on='customer_id', how='left')
unmatched = left[left['name'].isnull()]
print(len(unmatched))
print(len(unmatched) / len(sales) * 100)

country_stats = left.groupby('country_x')['total'].agg(['sum', 'mean', 'count']).reset_index()
category_stats = left.groupby('category').agg({'total': ['sum', 'mean'], 'quantity': ['mean']}).reset_index()
country_category_stats = left.groupby(["country_x", "category"])['total'].agg('sum').reset_index()

country_stats.to_csv("country_stats.csv", index=False)
category_stats.to_csv("category_stats.csv", index=False)
country_category_stats.to_csv("country_category_stats.csv", index=False)
left.to_csv("sales_with_customers.csv", index=False)

print(len(country_stats))
print(len(category_stats))
print(len(country_category_stats))
print(len(left))

sales_cols_num = sales.shape[1]
cust_cols_num = customers.shape[1]
expect_cols = sales_cols_num + cust_cols_num - 1
actual_cols = left.shape[1]
print(expect_cols)
print(actual_cols)

0
0.0
5
4
20
500
12
12
